# LASSI - Energy Efficiency

**NOTE:** This notebook has been pre-configured for an initial XSBench CUDA smoke test using the v3 pipeline.

Reference build/run script: `test_xsbench_cuda_build_run.py`


In [ ]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

os.chdir("/home/vchirio/Desktop/LASSI-EE")
print("CWD:", os.getcwd())


import lassi.llm_config
import lassi.pipeline_v3
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    raise SystemExit('Pipeline stopped - no GPU hardware found')
detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f'Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)')


In [ ]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open('source_apps.json', 'r') as f:
        source_apps = json.load(f)

    # ------------------------------
    # [1] SELECT SOURCE APP
    # ------------------------------
    # XSBench initial smoke test (CUDA)
    source_collection = 'Miniapps'
    source_app = 'XSBench'

    # Miniapps use Makefiles
    make_script = True
    build_script = False

    source_folder = source_apps[source_collection]['source_folder']
    app_folder_name = source_apps[source_collection][source_app]['app_folder']
    lines_of_code = source_apps[source_collection][source_app]['code_lines']

    # For XSBench, the kernel file is Main.cu (CUDA) or Main.cpp (HIP)
    app_name = source_app
    file_name = 'Main'

    # ------------------------------
    # [2] LANGUAGE + EXECUTION
    # ------------------------------
    source_lang = 'CUDA'
    target_lang = 'CUDA'

    test_code_first = True
    verifyprompt = False

    use_context_summaries_only = True
    few_shot_learning = False
    run_code_analysis = False

    # legacy flags (not used in v3)
    use_context = False
    use_self_prompting = False
    baseline_oneshot_goal = True

    power_measure = True
    vary_llm_temp = True
    yes_power_feedback = True

    # ------------------------------
    # [3] BUILD COMMAND
    # ------------------------------
    code_build_script = ''
    code_compiler = ''
    code_compiler_kwds = ''

    if source_lang == 'CUDA':
        # CUDA 13+ requires C++17+ for CCCL/Thrust/CUB/libcu++
        # Match the known-good smoke test: make clean; make -j SM_VERSION=80 CFLAGS=-std=c++17
        code_compiler = 'make'
        code_compiler_kwds = '-j SM_VERSION=80 CFLAGS=-std=c++17'
        context_knowledge_selection = 'cuda_optimization_summary' if yes_power_feedback else 'cuda'
    elif source_lang == 'HIP':
        code_compiler = 'make'
        code_compiler_kwds = '-j'
        context_knowledge_selection = 'hip_optimization_summary' if yes_power_feedback else 'hip'

    # ------------------------------
    # [4] EXECUTE
    # ------------------------------
    yes_execute = True
    exe_input_parameters = source_apps[source_collection][source_app]['app_params'] if yes_execute else ['']

    # ------------------------------
    # [5] LLM CONFIG (keep defaults)
    # ------------------------------
    model = 'gpto4mini'
    model_coder = 'gpto4mini'
    model_judge = 'gpt41'
    num_ctx = 200000
    num_ctx_coder = 200000

    if model in ['gpt5','gpt35','gpt35large','gpt4','gpt4large','gpt4turbo','gpt4o','gpt4olatest','gpt41',
                 'gpto1preview','gpto1mini','gpto3mini','gpto4mini','claudesonnet4']:
        llm_source = 'argo'
    elif model in ['gpt-oss-120b','qwen-3-coder-480b']:
        llm_source = 'cerebras'
    elif ('GGUF' in model and 'hf.co' not in model):
        llm_source = 'lmstudio'
    else:
        llm_source = 'ollama'

    experiment = f'exp_xsbench_smoketest_{model_coder}_{detected_vendor.upper()}'
    display_response = True
    test_with_error = False
    generate_test_case = False

    if llm_source == 'ollama':
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    DATAFIELDNAMES = [
        'trial_number','timestamp','experiment','gpu_vendor','app_name','code_line_count',
        'llm','llm_coder','llm_judge',
        'source_v0_energy_J','source_v0_avg_power_W','source_v0_runtime_s',
        'base_v1_energy_J','base_v1_avg_power_W','base_v1_runtime_s','base_v1_valid',
        'base_v1_energy_diff','base_v1_avg_power_diff','base_v1_runtime_diff',
        'lassiee_energy_J','lassiee_avg_power_W','lassiee_runtime_s','lassiee_valid',
        'lassiee_version','lassiee_llm_temp','lassiee_selfcorrections','lassiee_eeiterations',
        'lassiee_energy_diff','lassiee_avg_power_diff','lassiee_runtime_diff','lassiee_optimization_methods',
        'total_selfcorrections','total_eeiterations','pipeline_runtime_min'
    ]

    return (
        app_folder_name, app_name, file_name, lines_of_code,
        source_lang, target_lang,
        model, model_coder, model_judge,
        num_ctx, num_ctx_coder,
        llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis,
        context_knowledge_selection, source_folder,
        code_compiler, code_compiler_kwds,
        build_script, make_script, code_build_script,
        yes_execute, exe_input_parameters,
        experiment, display_response, test_with_error, generate_test_case,
        power_measure, test_code_first, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
        verifyprompt, DATAFIELDNAMES, detected_vendor.upper()
    )


In [ ]:
def init_data_collection(dir_path, DATAFIELDNAMES, filename='lassi-ee_pipeline_data.csv'):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp


In [ ]:
# Single trial smoke test
trial_number = 1
total_trials = 1

prompts = PromptDictionary()
all_trials_start = time.time()
original_home_directory = os.getcwd()

(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang,
    model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis,
    context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds,
    build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
    experiment, display_response, test_with_error, generate_test_case,
    power_measure, test_code_first, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
    verifyprompt, DATAFIELDNAMES, gpu_vendor
) = experimental_setup()

# Context knowledge
if gpu_vendor == 'NVIDIA':
    input_context = '[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] ' + prompts.get_codeknowledge('cuda') + '\n\n'
    input_context += '[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] ' + prompts.get_codeknowledge(str(context_knowledge_selection)) + '\n\n'
else:
    input_context = '[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] ' + prompts.get_codeknowledge('hip') + '\n\n'
    input_context += '[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] ' + prompts.get_codeknowledge(str(context_knowledge_selection)) + '\n\n'

system_prompt_selection_index = source_lang + '_ee'
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))
this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(source_lang + '_baseline_nocontext'))

# Load input code to refactor
input_code_folder = source_folder + '/' + app_folder_name + '/'
input_code_filename = file_name

if source_lang == 'HIP':
    input_code_extension = 'cpp'
else:
    input_code_extension = 'cu'

source_file_directory = 'source_codes/' + input_code_folder
source_file_path = os.path.abspath(source_file_directory + input_code_filename + '.' + input_code_extension)
with open(source_file_path, 'r') as f:
    input_code = f.read()

output_code_directory = 'generated_codes/' + source_folder + '/' + app_folder_name
output_code_file_name = input_code_filename

now = datetime.now(ZoneInfo('America/Chicago'))
dt_string = now.strftime('%m-%d-%Y_t%H_%M_%S')
experiment_data_directory = os.path.abspath('generated_codes/experiment/' + experiment + '__' + dt_string)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None)
experiment_data['gpu_vendor'] = str(gpu_vendor)

meta_data_file = '***** EXPERIMENT CONFIGURATION SUMMARY *****\n'
meta_data_file += '\nGPU Vendor: ' + str(gpu_vendor)
meta_data_file += '\nDate-time: ' + dt_string
meta_data_file += '\nApplication: ' + app_folder_name
meta_data_file += '\n\nLLM Source: ' + llm_source
meta_data_file += '\nModel: ' + model
meta_data_file += '\nModel coder: ' + model_coder
meta_data_file += '\nModel judge: ' + model_judge
meta_data_file += '\n\nPower measurement: ' + str(power_measure)
print(meta_data_file)

complete = lassi.pipeline_v3.auto_code_llmgeneration_pipeline(
    prompts,
    this_llm_system_prompt,
    this_llm_refactoredcode_prompt,
    llm_source,
    model,
    model_coder,
    model_judge,
    num_ctx,
    num_ctx_coder,
    use_context,
    use_context_summaries_only,
    few_shot_learning,
    run_code_analysis,
    input_context,
    '',
    input_code,
    lines_of_code,
    target_lang,
    input_code_extension,
    source_lang,
    code_compiler,
    code_compiler_kwds,
    build_script,
    make_script,
    code_build_script,
    output_code_directory,
    output_code_file_name,
    source_file_path,
    display_response,
    test_with_error,
    generate_test_case,
    yes_execute,
    exe_input_parameters,
    meta_data_file,
    power_measure,
    test_code_first,
    yes_power_feedback,
    dt_string,
    baseline_oneshot_goal,
    use_self_prompting,
    vary_llm_temp,
    experiment,
    verifyprompt,
    trial_number,
    experiment_data_directory,
    experiment_data_file_object,
    experiment_data,
    app_name
)

os.chdir(original_home_directory)
print('Pipeline completed' if complete else 'Pipeline failed')
print('Total time (min):', round((time.time() - all_trials_start) / 60, 2))
